In [1]:
# =====================================================================
# QUESTION 5.1
# OPTIMISATION OF THE STRONGEST VALIDATION MODEL
# =====================================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    RepeatedStratifiedKFold,
    GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression

SEED = 42


# =====================================================================
# 1. LOAD DATASET
# =====================================================================

PATH = r"C:\Users\Sekomane\Desktop\Data ScienceTUT\NIWIS_Water Supply Reliability - Population_26-Sep-2026.csv"

df = pd.read_csv(PATH)

print("=" * 75)
print("QUESTION 5.1 - LOGISTIC REGRESSION OPTIMISATION")
print("=" * 75)

print("Original dataset shape:", df.shape)


# =====================================================================
# 2. CLEAN DATA
# =====================================================================

data = df.copy()

# Remove empty unnamed columns
data = data.loc[:, ~data.columns.str.contains("^Unnamed")]

# Standardise column names
data.columns = (
    data.columns
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
)


def clean_number(series):
    """
    Convert population/count values to numeric.
    Removes spaces and commas before conversion.
    """

    return pd.to_numeric(
        series.astype(str)
              .str.replace(" ", "", regex=False)
              .str.replace(",", "", regex=False)
              .str.strip(),
        errors="coerce"
    )


# Convert population/count columns
for col in data.columns:

    if "%" not in col and col != "WSA Name":

        data[col] = clean_number(
            data[col]
        )


# Convert percentage columns
for col in data.columns:

    if "%" in col:

        data[col] = pd.to_numeric(
            data[col]
                .astype(str)
                .str.replace("%", "", regex=False)
                .str.strip(),
            errors="coerce"
        )


# =====================================================================
# 3. CREATE TARGET
# =====================================================================

# Class 0 = Reliable supply (>= 60%)
# Class 1 = Low reliability (< 60%)

data["low_reliability"] = (
    data["% Population Reliable Supply"] < 60
).astype(int)


print("\nTarget distribution:")

print(
    data["low_reliability"]
        .value_counts()
        .sort_index()
)


# =====================================================================
# 4. REMOVE LEAKAGE VARIABLES
# =====================================================================

leakage_columns = [

    "Population Reliable Supply",
    "% Population Reliable Supply",

    "Urban Population Reliable Supply",
    "% Urban Population Reliable Supply",

    "Rural Population Reliable Supply",
    "% Rural Population Reliable Supply"
]


y = data["low_reliability"]


X = data.drop(
    columns=[
        "low_reliability",
        "WSA Name"
    ] + leakage_columns,
    errors="ignore"
).copy()


# =====================================================================
# 5. FEATURE ENGINEERING
# =====================================================================

X["population_access_rate"] = (
    X["Population with access to water"]
    / X["Population"]
) * 100


X["urban_access_rate"] = (
    X["Urban Population with access to water"]
    / X["Urban Population"]
) * 100


X["rural_access_rate"] = (
    X["Rural Population with access to water"]
    / X["Rural Population"]
) * 100


X["urban_population_share"] = (
    X["Urban Population"]
    / X["Population"]
) * 100


X["rural_population_share"] = (
    X["Rural Population"]
    / X["Population"]
) * 100


# Handle divisions by zero
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)


# =====================================================================
# 6. DEVELOPMENT / TEST SPLIT
# =====================================================================

# IMPORTANT:
# The test set is created here but is NOT used during optimisation.
# Hyperparameter tuning is performed ONLY using X_dev and y_dev.

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)


print("\n" + "=" * 75)
print("DATA SPLIT")
print("=" * 75)

print("Development samples :", len(X_dev))
print("Held-out test samples:", len(X_test))

print("\nDevelopment target distribution:")

print(
    y_dev.value_counts().sort_index()
)

print(
    "\nHeld-out test set created but NOT used for hyperparameter tuning."
)


# =====================================================================
# 7. LOGISTIC REGRESSION PIPELINE
# =====================================================================

logistic_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "scaler",
        RobustScaler()
    ),

    (
        "classifier",
        LogisticRegression(
            random_state=SEED,
            max_iter=5000
        )
    )
])


# =====================================================================
# 8. DEFINE HYPERPARAMETER SEARCH SPACE
# =====================================================================

# C:
# Controls regularisation strength.
#
# Smaller C = stronger regularisation.
# Larger C  = weaker regularisation.
#
# class_weight:
# Tests whether balanced class weighting improves performance.
#
# solver:
# We compare two solvers compatible with L1/L2 penalties.

param_grid = [

    # -------------------------------------------------------------
    # L2 regularisation
    # -------------------------------------------------------------

    {
        "classifier__C": [
            0.001,
            0.01,
            0.1,
            1,
            10,
            100
        ],

        "classifier__penalty": [
            "l2"
        ],

        "classifier__solver": [
            "liblinear",
            "saga"
        ],

        "classifier__class_weight": [
            None,
            "balanced"
        ]
    },


    # -------------------------------------------------------------
    # L1 regularisation
    # -------------------------------------------------------------

    {
        "classifier__C": [
            0.001,
            0.01,
            0.1,
            1,
            10,
            100
        ],

        "classifier__penalty": [
            "l1"
        ],

        "classifier__solver": [
            "liblinear",
            "saga"
        ],

        "classifier__class_weight": [
            None,
            "balanced"
        ]
    }
]


# =====================================================================
# 9. CROSS-VALIDATION STRATEGY
# =====================================================================

# Same repeated stratified CV design used during model comparison.

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=SEED
)


# =====================================================================
# 10. GRID SEARCH
# =====================================================================

# Macro-F1 is the scoring rule because both classes should contribute
# equally to model selection despite the class imbalance.

grid_search = GridSearchCV(

    estimator=logistic_pipeline,

    param_grid=param_grid,

    scoring="f1_macro",

    cv=cv,

    n_jobs=-1,

    refit=True,

    return_train_score=True
)


print("\n" + "=" * 75)
print("STARTING GRID SEARCH")
print("=" * 75)

print("\nOptimisation model: Logistic Regression")
print("Search method     : GridSearchCV")
print("Scoring rule      : Macro-F1")
print("CV strategy       : Repeated Stratified 5-Fold CV (3 repeats)")

print("\nRunning optimisation...")


# IMPORTANT:
# Only development data is supplied here.

grid_search.fit(
    X_dev,
    y_dev
)


# =====================================================================
# 11. BEST RESULT
# =====================================================================

print("\n" + "=" * 75)
print("QUESTION 5.1 - OPTIMISATION RESULTS")
print("=" * 75)


print(
    "\nBest CV Macro-F1:",
    round(
        grid_search.best_score_,
        3
    )
)


print("\nBest hyperparameters:")


for parameter, value in grid_search.best_params_.items():

    print(
        f"{parameter}: {value}"
    )


# =====================================================================
# 12. SHOW TOP SEARCH RESULTS
# =====================================================================

search_results = pd.DataFrame(
    grid_search.cv_results_
)


top_results = search_results[[
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "param_classifier__C",
    "param_classifier__penalty",
    "param_classifier__solver",
    "param_classifier__class_weight"
]].copy()


top_results = top_results.sort_values(
    "rank_test_score"
).head(10)


top_results = top_results.rename(
    columns={
        "rank_test_score":
            "Rank",

        "mean_test_score":
            "Mean CV Macro-F1",

        "std_test_score":
            "CV Macro-F1 SD",

        "mean_train_score":
            "Mean Training Macro-F1",

        "param_classifier__C":
            "C",

        "param_classifier__penalty":
            "Penalty",

        "param_classifier__solver":
            "Solver",

        "param_classifier__class_weight":
            "Class Weight"
    }
)


top_results[
    [
        "Mean CV Macro-F1",
        "CV Macro-F1 SD",
        "Mean Training Macro-F1"
    ]
] = (
    top_results[
        [
            "Mean CV Macro-F1",
            "CV Macro-F1 SD",
            "Mean Training Macro-F1"
        ]
    ].round(3)
)


print("\nTop 10 hyperparameter combinations:")

display(
    top_results.reset_index(drop=True)
)


# =====================================================================
# 13. SAVE OPTIMISED MODEL IN MEMORY
# =====================================================================

optimized_logistic_model = (
    grid_search.best_estimator_
)


print("\n" + "=" * 75)

print(
    "QUESTION 5.1 COMPLETED SUCCESSFULLY"
)

print("=" * 75)


print(
    "\nThe held-out test set was NOT used during optimisation."
)

QUESTION 5.1 - LOGISTIC REGRESSION OPTIMISATION
Original dataset shape: (144, 14)

Target distribution:
low_reliability
0    105
1     39
Name: count, dtype: int64

DATA SPLIT
Development samples : 115
Held-out test samples: 29

Development target distribution:
low_reliability
0    84
1    31
Name: count, dtype: int64

Held-out test set created but NOT used for hyperparameter tuning.

STARTING GRID SEARCH

Optimisation model: Logistic Regression
Search method     : GridSearchCV
Scoring rule      : Macro-F1
CV strategy       : Repeated Stratified 5-Fold CV (3 repeats)

Running optimisation...

QUESTION 5.1 - OPTIMISATION RESULTS

Best CV Macro-F1: 0.833

Best hyperparameters:
classifier__C: 1
classifier__class_weight: None
classifier__penalty: l1
classifier__solver: liblinear

Top 10 hyperparameter combinations:


D:\Sekomane\Documents\anaconda\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
D:\Sekomane\Documents\anaconda\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,Rank,Mean CV Macro-F1,CV Macro-F1 SD,Mean Training Macro-F1,C,Penalty,Solver,Class Weight
0,1,0.833,0.057,0.841,1.0,l1,liblinear,NaN
1,2,0.826,0.042,0.843,1.0,l1,saga,NaN
2,3,0.824,0.036,0.849,1.0,l2,liblinear,NaN
3,4,0.823,0.064,0.849,1.0,l2,saga,balanced
4,5,0.822,0.054,0.842,0.1,l1,liblinear,NaN
5,6,0.820,0.051,0.831,0.1,l2,liblinear,NaN
6,7,0.820,0.069,0.852,1.0,l2,liblinear,balanced
7,8,0.819,0.034,0.843,1.0,l2,saga,NaN
8,9,0.816,0.047,0.857,100.0,l1,saga,balanced
9,9,0.816,0.047,0.858,100.0,l2,saga,balanced



QUESTION 5.1 COMPLETED SUCCESSFULLY

The held-out test set was NOT used during optimisation.
